# Week 2 Wednesday - Discrete Log

In [3]:
import numpy as np
import pandas as pd
import altair as alt

from sympy.ntheory import isprime, nextprime, factorint, n_order

Why is discrete log so much harder to compute than ordinary logarithm?

## Properties of real-number logarithms

In [5]:
help(np.emath.logn)

Help on _ArrayFunctionDispatcher in module numpy.lib.scimath:

logn(n, x)
    Take log base n of x.

    If `x` contains negative inputs, the answer is computed and returned in the
    complex domain.

    Parameters
    ----------
    n : array_like
       The integer base(s) in which the log is taken.
    x : array_like
       The value(s) whose log base `n` is (are) required.

    Returns
    -------
    out : ndarray or scalar
       The log base `n` of the `x` value(s). If `x` was a scalar, so is
       `out`, otherwise an array is returned.

    Examples
    --------
    >>> np.set_printoptions(precision=4)

    >>> np.emath.logn(2, [4, 8])
    array([2., 3.])
    >>> np.emath.logn(2, [-4, -8, 8])
    array([2.+4.5324j, 3.+4.5324j, 3.+0.j    ])



In [7]:
# Easier to type
logn = np.emath.logn

In [7]:
logn(17, 324723874328247)

11.793674623824684

In [9]:
%%time
logn(17, 324723874328247)

CPU times: total: 0 ns
Wall time: 0 ns


11.793674623824684

In [13]:
%%time
logn(17, 3247238743282470000)

CPU times: total: 0 ns
Wall time: 0 ns


15.04452066099252

Properties of logs tell us $\log_{17} 3247238743282470000 = \log_{17} 324723874328247 + \log_{17} 10000$, so the added time should be negligible.

We will graph $y = 17^x$ using Altair

In [16]:
x = np.linspace(0, 6, 500) #500 pts from 0 to 6

In [18]:
x[:5] #Sample of 1st 5 pts

array([0.        , 0.01202405, 0.0240481 , 0.03607214, 0.04809619])

In [20]:
# array broadcasting
y = 17**x  #note ** is used for exponentiation in Python.

In [22]:
y2 = pow(17, x) #The pow function also broadcasts across x

In [24]:
y[:5]

array([1.        , 1.03465361, 1.07050809, 1.10760506, 1.14598757])

In [26]:
df = pd.DataFrame({
    "x": x,
    "y": y
})

In [28]:
df[:5]

,x,y
0,0.000000,1.000000
1,0.012024,1.034654
2,0.024048,1.070508
3,0.036072,1.107605
4,0.048096,1.145988


In [30]:
# Notice how regular this graph is
alt.Chart(df).mark_circle().encode(
    x = "x",
    y = "y"
)

alt.Chart(...)

## Primitive roots and Discrete Log

In [55]:
nextprime(20)

23

In [206]:
p = nextprime(1000)
p

1009

Let's try to find a primitive root modulo 1009.

In [60]:
# Is 2 a primitive root?
n_order(2, 1009)

504

In [62]:
n_order(3, 1009)

168

In [64]:
n_order(4, 1009)

252

In [66]:
n_order(5, p)

504

In [68]:
for g in range(1, p):
    if n_order(g,p) == p-1:
        break

In [70]:
g

11

In [208]:
x = range(p)
y = [pow(g, a, p) for a in x] # pow with three arguments is used for efficient modular exponentiation
# Much more efficient for big values than pow(g,a)%p
#Sadly, we cannot broadcast pow(g,x,p), even if we use x = list(range(p)) or x = np.arange(p). Therefore, we use list comprehension

In [210]:
df_discrete = pd.DataFrame({
    "x": x,
    f"y = 11^x mod {p}": y
})

In [212]:
df_discrete[:5]

,x,y = 11^x mod 1009
0,0,1
1,1,11
2,2,121
3,3,322
4,4,515


In [214]:
print(11**3)
print((11**3)%1009)

1331
322


In [216]:
# Notice how irregular this looks
alt.Chart(df_discrete).mark_circle().encode(
    x = "x",
    y = f"y = 11^x mod {p}",
    tooltip = ["x", f"y = 11^x mod {p}"]
)

alt.Chart(...)

Because this is so irregular (no obvious patterns), our intuition is that it's difficult to invert this procedure, i.e., discrete logarithms are difficult to compute

## Discrete log when our base is not a primitive root

In [222]:
p = 65537
x = range(5000)
y = [pow(2, a, p) for a in x]

In [224]:
myleg = f"y = 2^x mod {p}"
df2 = pd.DataFrame(
    {
        "x": x,
        myleg: y
    }
)

In [226]:
df2[:5]

,x,y = 2^x mod 65537
0,0,1
1,1,2
2,2,4
3,3,8
4,4,16


In [228]:
alt.Chart(df2).mark_circle().encode(
    x = "x",
    y = myleg,
    tooltip = ["x", myleg]
)

alt.Chart(...)

In [230]:
# What went wrong?  2 has a very small multiplicative order modulo p
n_order(2, p)

32

In [232]:
2**32

4294967296

In [234]:
2**16

65536

In [236]:
p

65537

Therefore, to make the discrete log problem as difficult as possible, we must pick a primitive root mod p for our base.

## Discrete log timing

### Method 1: Brute Force. 

We use naive powering to take every power of $g$ and check if $g^x \equiv h \textnormal{ mod } p$.  The expected number of steps has order $\mathcal{O}(p)$ in the worst case scenario.

In [97]:
def DL_Brute(base, target, p, verbose = False):
    '''
    Take every power x of base for x between 0 and p-1. Check whether ((base**r) % p) == (target %p). Return x if a solution is found, and None otherwise.
    '''
    a = 1
    h = 12
    if verbose == True:
        print(f"exponent {0}: {base}^{0} = {a} mod {p}")
    for i in range(p-1):
        a = (a * base) % p
        if verbose == True:
            print(f"exponent {i+1}: {base}^{i+1} = {a} mod {p}")
        if a == h:
            print(f"found solution, exponent = {i+1}")
            return (i+1)
    print("No solution found")
    return None

In [99]:
DL_Brute(3, 12, 401, verbose = True)

exponent 0: 3^0 = 1 mod 401
exponent 1: 3^1 = 3 mod 401
exponent 2: 3^2 = 9 mod 401
exponent 3: 3^3 = 27 mod 401
exponent 4: 3^4 = 81 mod 401
exponent 5: 3^5 = 243 mod 401
exponent 6: 3^6 = 328 mod 401
exponent 7: 3^7 = 182 mod 401
exponent 8: 3^8 = 145 mod 401
exponent 9: 3^9 = 34 mod 401
exponent 10: 3^10 = 102 mod 401
exponent 11: 3^11 = 306 mod 401
exponent 12: 3^12 = 116 mod 401
exponent 13: 3^13 = 348 mod 401
exponent 14: 3^14 = 242 mod 401
exponent 15: 3^15 = 325 mod 401
exponent 16: 3^16 = 173 mod 401
exponent 17: 3^17 = 118 mod 401
exponent 18: 3^18 = 354 mod 401
exponent 19: 3^19 = 260 mod 401
exponent 20: 3^20 = 379 mod 401
exponent 21: 3^21 = 335 mod 401
exponent 22: 3^22 = 203 mod 401
exponent 23: 3^23 = 208 mod 401
exponent 24: 3^24 = 223 mod 401
exponent 25: 3^25 = 268 mod 401
exponent 26: 3^26 = 2 mod 401
exponent 27: 3^27 = 6 mod 401
exponent 28: 3^28 = 18 mod 401
exponent 29: 3^29 = 54 mod 401
exponent 30: 3^30 = 162 mod 401
exponent 31: 3^31 = 85 mod 401
exponent 32:

53

In [101]:
DL_Brute(5, 12, 401)

No solution found


In [133]:
p = nextprime(10**7)
print(p)
DL_Brute(5,12,p)

10000019
found solution, exponent = 3866671


3866671

In [134]:
p = nextprime(10**8)
print(p)
DL_Brute(5,12,p)

100000007
found solution, exponent = 15137028


15137028

In [137]:
from time import time

In [141]:
t = 0
n = 100
while t < 60:
    p = nextprime(n)
    t1 = time()
    DL_Brute(5,12,p)
    t2 = time()
    t = t2 - t1
    print(f"prime: {p}, time = {t}")
    n = 10*n

No solution found
prime: 101, time = 0.0
found solution, exponent = 299
prime: 1009, time = 0.0
found solution, exponent = 9340
prime: 10007, time = 0.003813028335571289
found solution, exponent = 56943
prime: 100003, time = 0.02292323112487793
found solution, exponent = 884389
prime: 1000003, time = 0.44783735275268555
found solution, exponent = 3866671
prime: 10000019, time = 1.5584545135498047
found solution, exponent = 15137028
prime: 100000007, time = 6.45593523979187
found solution, exponent = 647914256
prime: 1000000007, time = 320.9372215270996


In [144]:
647914256/15137028

42.80326732565997

In [146]:
320.9372215270996/6.45593523979187

49.711964201401464

In [148]:
3866671/884389

4.372138278517711

In [150]:
1.5584545135498047/0.44783735275268555

3.4799565153968928

Moral: Worst-case scaling is based on the size of p, but not all solutions are found in worst-case timing.

### Method 2: Trial and Error

We pick an exponent x at random from 0 to p-1, and use fast powering to check if $g^x \equiv h \textnormal{ mod } p$.  The expected number of steps has order $\mathcal{O}\left((\log_2{x})^2p\right)$ in the worst case scenario. Our attempt to use fast powering by trial and error should be worse than brute force on average!

We'll leave this as an exercise for now. I've included the fast-powering function from Week 1. Remember to sample powers without replacement!

In [9]:
def fast_powering(base, exponent, modulo):
    pow_limit = np.floor(np.emath.logn(2, exponent))
    a = base
    pow_list = [a]
    for i in range(int(pow_limit)):
        a = a*a
        a = a%modulo
        pow_list.append(a)
    current_exp = exponent - 2**pow_limit
    pow_list.pop(-1)
    pows_left = 2**np.arange(pow_limit-1,-1,-1)
    for x in pows_left:
        if current_exp >= x:
            a = a*pow_list[-1]
            a = a%modulo
            current_exp = current_exp-x
        pow_list.pop(-1)
    return a

### Method 3: Shank's Collision Algorithm

See The Week 3 python notes for more details!